<a href="https://colab.research.google.com/github/nermeenmansourr/RFM_Analysis/blob/main/RFM_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [42]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from datetime import datetime

In [43]:
#loading the transactions datasets
cust_demo=pd.read_csv('CustomerDemographic_Cleaned.csv')
trans=pd.read_csv('Transactions_Cleaned.csv')

In [44]:
#fetching the first 5 records
trans.head(5)

,transaction_id,product_id,customer_id,transaction_date,online_order,order_status,brand,product_line,product_class,product_size,list_price,standard_cost,product_first_sold_date,Profit
0,1,2,2950,42791,0.0,Approved,Solex,Standard,medium,medium,71.49,53.62,41245.0,17.87
1,2,3,3120,42876,1.0,Approved,Trek Bicycles,Standard,medium,large,2091.47,388.92,41701.0,1702.55
2,3,37,402,43024,0.0,Approved,OHM Cycles,Standard,low,medium,1793.43,248.82,36361.0,1544.61
3,4,88,3135,42978,0.0,Approved,Norco Bicycles,Standard,medium,medium,1198.46,381.10,36145.0,817.36
4,5,78,787,43009,1.0,Approved,Giant Bicycles,Standard,medium,large,1765.30,709.48,42226.0,1055.82


In [45]:
# total records for transactions table
print('Total records (rows):{}'.format(trans.shape[0]))
print('Total records (columns):{}'.format(trans.shape[1]))

Total records (rows):19803
Total records (columns):14


In [46]:
#fetching 5 customers demographic field
cust_demo.head(5)

,customer_id,first_name,last_name,gender,past_3_years_bike_related_purchases,DOB,job_title,job_industry_category,wealth_segment,deceased_indicator,owns_car,tenure,Age
0,1,Laraine,Medendorp,Female,93,1953-10-12,Executive Secretary,Health,Mass Customer,N,Yes,11.0,67
1,2,Eli,Bockman,Male,81,1980-12-16,Administrative Officer,Financial Services,Mass Customer,N,Yes,16.0,40
2,3,Arlin,Dearle,Male,61,1954-01-20,Recruiting Manager,Property,Mass Customer,N,Yes,15.0,67
3,4,Talbot,NaN,Male,33,1961-10-03,Missing,IT,Mass Customer,N,No,7.0,59
4,5,Sheila-kathryn,Calton,Female,56,1977-05-13,Senior Editor,Missing,Affluent Customer,N,Yes,8.0,44


In [47]:
# total records for customer demographic table
print('Total records (rows):{}'.format(cust_demo.shape[0]))
print('Total records (columns):{}'.format(cust_demo.shape[1]))

Total records (rows):3912
Total records (columns):13


In [48]:
# Merging both the Transaction Dataset and Customer Demographics Dataset based on customer_id.
cust_trans=pd.merge(trans,cust_demo,on='customer_id',how='inner')

In [49]:
#fetching the first 5 records after joining
cust_trans.head(5)

,transaction_id,product_id,customer_id,transaction_date,online_order,order_status,brand,product_line,product_class,product_size,...,gender,past_3_years_bike_related_purchases,DOB,job_title,job_industry_category,wealth_segment,deceased_indicator,owns_car,tenure,Age
0,1,2,2950,42791,0.0,Approved,Solex,Standard,medium,medium,...,Male,19,1955-01-11,Software Engineer I,Financial Services,Mass Customer,N,Yes,10.0,66
1,2,3,3120,42876,1.0,Approved,Trek Bicycles,Standard,medium,large,...,Female,89,1979-02-04,Clinical Specialist,Health,Mass Customer,N,Yes,10.0,42
2,3,37,402,43024,0.0,Approved,OHM Cycles,Standard,low,medium,...,Male,9,1977-06-03,Desktop Support Technician,Retail,Affluent Customer,N,No,22.0,43
3,4,88,3135,42978,0.0,Approved,Norco Bicycles,Standard,medium,medium,...,Male,83,1962-01-14,Staff Scientist,Financial Services,Mass Customer,N,No,16.0,59
4,5,78,787,43009,1.0,Approved,Giant Bicycles,Standard,medium,large,...,Female,29,1996-12-15,Missing,Retail,Affluent Customer,N,Yes,2.0,24


In [50]:
cust_trans.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19354 entries, 0 to 19353
Data columns (total 26 columns):
 #   Column                               Non-Null Count  Dtype  
---  ------                               --------------  -----  
 0   transaction_id                       19354 non-null  int64  
 1   product_id                           19354 non-null  int64  
 2   customer_id                          19354 non-null  int64  
 3   transaction_date                     19354 non-null  int64  
 4   online_order                         19004 non-null  float64
 5   order_status                         19354 non-null  object 
 6   brand                                19354 non-null  object 
 7   product_line                         19354 non-null  object 
 8   product_class                        19354 non-null  object 
 9   product_size                         19354 non-null  object 
 10  list_price                           19354 non-null  float64
 11  standard_cost               

# The data-type of transaction_date column is not in date-time format. Hence the data-type of the column should be changed from object to datetime type.

In [51]:

cust_trans['transaction_date'] = pd.to_datetime(
    pd.to_numeric(cust_trans['transaction_date'], errors='coerce'),
    unit='D',
    origin='1899-12-30'
)

In [52]:
cust_trans['transaction_date']

,transaction_date
0,2017-02-25
1,2017-05-21
2,2017-10-16
3,2017-08-31
4,2017-10-01
...,...
19349,2017-06-24
19350,2017-11-09
19351,2017-04-14
19352,2017-07-03


# 1. RFM Analysis
RFM Analysis is a behavioral customer segmentation method that groups customers based on their historical purchase transactions across three dimensions:

*   Recency (R): Measures how recently a customer made a purchase (typically days since the last transaction; lower values indicate more recent activity).
 *  Frequency (F): Measures how often a customer purchases (total number of transactions over a given period; higher values indicate stronger engagement).

* Monetary (M): Measures how much a customer spends (total revenue or spend generated; higher values indicate greater financial contribution).

In [53]:
#we need to find the latest date so we can compare to
max_date=max(cust_trans['transaction_date']).date()

In [54]:
max_date

datetime.date(2017, 12, 30)

In [55]:
#find the days between the last transaction date and transaction date for every customer ,it will be comparison date


In [56]:
comparison_date=datetime.strptime(str(max_date),'%Y-%m-%d')

In [57]:
comparison_date

datetime.datetime(2017, 12, 30, 0, 0)

In [58]:
#creating rfm table that will contain all the info for recency, frequency, monetary
rfm_table = cust_trans.groupby('customer_id').agg({
    'transaction_date': lambda date: (comparison_date-date.max()).days,
    'product_id': lambda prod_id: len(prod_id),
    'Profit': lambda profit: sum(profit)
})

# Rename columns for clarity
rfm_table.rename(columns={
    'transaction_date': 'Recency',
    'product_id': 'Frequency',
    'Profit': 'Monetary'
}, inplace=True)

rfm_table.head(5)

,Recency,Frequency,Monetary
customer_id,,,
1,7,11,3018.09
2,128,3,2226.26
3,102,8,3362.81
4,195,2,220.57
5,16,6,2394.94


In [59]:
# Dividing the recency, frequency and monetary into 4 quartiles (min, 25%, 50%, 75% and max).
# These values will help us to calculate RFM score for a customer and classify based on their RFM score.
# Note: duplicates='drop' is used to handle situations where there are identical values preventing 4 unique bins.

rfm_table['r_quartile']=pd.qcut(rfm_table['Recency'],4,['4','3','2','1'])
rfm_table['f_quartile']=pd.qcut(rfm_table['Frequency'],4,['4','3','2','1'])
rfm_table['m_quartile']=pd.qcut(rfm_table['Monetary'],4,['4','3','2','1'])

In [60]:
#rfm dataset
rfm_table

,Recency,Frequency,Monetary,r_quartile,f_quartile,m_quartile
customer_id,,,,,,
1,7,11,3018.09,4,1,2
2,128,3,2226.26,1,4,3
3,102,8,3362.81,1,1,2
4,195,2,220.57,1,4,4
5,16,6,2394.94,4,3,3
...,...,...,...,...,...,...
3496,256,4,2045.84,1,4,3
3497,52,3,1648.32,2,4,4
3498,127,6,3147.33,1,3,2


In [61]:
#rfm score
rfm_table['RFM_score'] = (
    rfm_table['r_quartile'].astype(int)
    + rfm_table['f_quartile'].astype(int)
    + rfm_table['m_quartile'].astype(int)
)

In [62]:
rfm_table

,Recency,Frequency,Monetary,r_quartile,f_quartile,m_quartile,RFM_score
customer_id,,,,,,,
1,7,11,3018.09,4,1,2,7
2,128,3,2226.26,1,4,3,8
3,102,8,3362.81,1,1,2,4
4,195,2,220.57,1,4,4,9
5,16,6,2394.94,4,3,3,10
...,...,...,...,...,...,...,...
3496,256,4,2045.84,1,4,3,8
3497,52,3,1648.32,2,4,4,10
3498,127,6,3147.33,1,3,2,6


In [63]:
#segment the customers
rfm_table['customer_segment'] = pd.qcut(rfm_table['RFM_score'], q=4, labels=['Bronze', 'Silver', 'Gold', 'Platinum'])

In [64]:
rfm_table

,Recency,Frequency,Monetary,r_quartile,f_quartile,m_quartile,RFM_score,customer_segment
customer_id,,,,,,,,
1,7,11,3018.09,4,1,2,7,Silver
2,128,3,2226.26,1,4,3,8,Silver
3,102,8,3362.81,1,1,2,4,Bronze
4,195,2,220.57,1,4,4,9,Gold
5,16,6,2394.94,4,3,3,10,Platinum
...,...,...,...,...,...,...,...,...
3496,256,4,2045.84,1,4,3,8,Silver
3497,52,3,1648.32,2,4,4,10,Platinum
3498,127,6,3147.33,1,3,2,6,Bronze


In [65]:
#merging rfm table with customers transactions to gain deeper insights
cust_trans_rfm=pd.merge(cust_trans,rfm_table,on='customer_id',how='inner'  )

In [66]:
#fetching the first 5 records
cust_trans_rfm.head(5)

,transaction_id,product_id,customer_id,transaction_date,online_order,order_status,brand,product_line,product_class,product_size,...,tenure,Age,Recency,Frequency,Monetary,r_quartile,f_quartile,m_quartile,RFM_score,customer_segment
0,1,2,2950,2017-02-25,0.0,Approved,Solex,Standard,medium,medium,...,10.0,66,75,3,645.99,2,4,4,10,Platinum
1,2,3,3120,2017-05-21,1.0,Approved,Trek Bicycles,Standard,medium,large,...,10.0,42,20,7,4179.11,3,2,1,6,Bronze
2,3,37,402,2017-10-16,0.0,Approved,OHM Cycles,Standard,low,medium,...,22.0,43,56,6,4965.43,2,3,1,6,Bronze
3,4,88,3135,2017-08-31,0.0,Approved,Norco Bicycles,Standard,medium,medium,...,16.0,59,121,7,3983.97,1,2,2,5,Bronze
4,5,78,787,2017-10-01,1.0,Approved,Giant Bicycles,Standard,medium,large,...,2.0,24,75,10,6646.00,2,1,1,4,Bronze


In [67]:
cust_trans_rfm.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19354 entries, 0 to 19353
Data columns (total 34 columns):
 #   Column                               Non-Null Count  Dtype         
---  ------                               --------------  -----         
 0   transaction_id                       19354 non-null  int64         
 1   product_id                           19354 non-null  int64         
 2   customer_id                          19354 non-null  int64         
 3   transaction_date                     19354 non-null  datetime64[ns]
 4   online_order                         19004 non-null  float64       
 5   order_status                         19354 non-null  object        
 6   brand                                19354 non-null  object        
 7   product_line                         19354 non-null  object        
 8   product_class                        19354 non-null  object        
 9   product_size                         19354 non-null  object        
 10  list_price

no issue with the datatypes

creating a age feature group

In [68]:
age_group = pd.cut(cust_trans_rfm['Age'],
                   bins=[0, 10, 29, 39, 100],
                   labels=['20', '30', '40', '50+'])

In [69]:
#rfm score
bins = [0, 111, 123, 211, 223, 310, 322, 343, 420, 432, 443, 999]
labels = [
    'Lost Customer',
    'Evasive Customer',
    'Almost Lost Customer',
    'High Risk Customer',
    'Loosing Customer',
    'Late Bloomer',
    'Potential Customer',
    'Recent Customer',
    'Becoming Loyal',
    'Very Loyal',
    'Platinum Customer',
]
rfm_table['customer_title']=pd.cut(
    rfm_table['RFM_score'].astype('int'),
    bins=bins,
    labels=labels,
    right=True
               )


In [70]:
#adding customer_title to cust_trans_rfm table
title_lookup=rfm_table['customer_title']
cust_trans_rfm['detailed_customer_title']=cust_trans_rfm['customer_id'].map(title_lookup)

In [71]:
#provide rank to every customer based on theier title
rank_map = {
    'Lost Customer': 11,
    'Evasive Customer': 10,
    'Almost Lost Customer': 9,
    'High Risk Customer': 8,
    'Loosing Customer': 7,
    'Late Bloomer': 6,
    'Potential Customer': 5,
    'Recent Customer': 4,
    'Becoming Loyal': 3,
    'Very Loyal': 2,
    'Platinum Customer': 1,
}

rfm_table['rank'] = rfm_table['customer_title'].map(rank_map)

In [72]:
#adding rank column to cust_trans_rfm table
rank_lookup=rfm_table['rank']
cust_trans_rfm['Rank']=cust_trans_rfm['customer_id'].map(rank_lookup)

# 2. Exporting to CSV File
After performing data quality assessment(DQA), data cleaning and RFM Analysis on the dataset, it's time to export the dataset to a csv file for further exploratory data analysis (EDA) and this data will drive the Sales Customer Segmenation Dashboard developed in python



In [73]:
cust_trans_rfm.to_csv('customer_trans_rfm_analysis.csv',index=False)